# Single-pass learnable VQ query

Trains the learnable vector-quantized query of `StableQwen3TTSForConditionalGeneration`
and generates with it in one autoregressive roll.

Two things differ from the earlier two-stage runs:

1. **No text commit.** The query never leaves embedding space. Earlier runs decoded
   `token_ids` back into a string and passed it as the stage-2 `ref_text`, which put
   unreadable tokens into the same text channel the instruction occupies. That is the
   text-space collision the 2026.07.27 material flagged as a likely cause of the
   InstructTTSEval regression on DSD and RP.
2. **No stage split.** The anchor is the first `ANCHOR_FRAMES` frames of the same roll,
   under one key value cache. There is no reference audio to re-encode and therefore no
   `ref_text` to match, which is what forced the earlier WER trade-off.

Prompt augmentation renders each LibriTTS-P sample in the registers InstructTTSEval
actually uses (APS, DSD, RP) instead of only the LibriTTS-P persona register.

## Environment

In [ ]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
SEED = int(os.environ.get("RUN_SEED", "1"))
RUN_TAG = os.environ.get("RUN_TAG", f"s{SEED}")
RUN_LR = float(os.environ.get("RUN_LR", "0.02"))
RUN_PROJECT = os.environ.get("RUN_PROJECT", "0") == "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

In [ ]:
import json
import math
import random
import time
import types
from pathlib import Path

import librosa
import numpy as np
import soundfile
import torch
import torch.nn.functional as F

from spk_incon.datasets import LIBRITTS_P_Custom
from spk_incon.datasets.libritts_p3 import download_libritts_p_metadata
from spk_incon.datasets.prompt_views import PromptView, build_prompt
from spk_incon.models.stable_qwen3_tts import (
    StableQwen3TTSConfig,
    StableQwen3TTSForConditionalGeneration,
)
from voicestudio.models.qwen3_tts import Qwen3TTSProcessor

## Configuration

In [ ]:
# The published Qwen checkpoint is in the original layout. transformers 5.16 expects the
# flattened layout, so point this at the converted copy produced by
# convert_qwen3_tts_to_hf.py plus the merged code_predictor.lm_head.
MODEL_ID = "ckpt/Qwen3-TTS-12Hz-1.7B-VoiceDesign-HF"
DEVICE = torch.device("cuda:0")
DTYPE = torch.bfloat16

NUM_QUERY_TOKENS = 32
ANCHOR_FRAMES = 32
SUB_TALKER_WEIGHT = 0.3

NUM_PAIRS = 1600
BATCH_SIZE = 32
MAX_BATCH_TOKENS = 4608
STEPS = 300
LEARNING_RATE = RUN_LR
SCHEDULE = "cosine"
ATTN_IMPLEMENTATION = "flash_attention_2"

MIN_CODEC_FRAMES, MAX_CODEC_FRAMES = 4, 400
VIEW_WEIGHTS = {PromptView.RAW: 1 / 3, PromptView.APS: 1 / 4, PromptView.DSD: 1 / 4, PromptView.RP: 1 / 6}

CKPT_DIR = Path("ckpt")
CKPT_PATH = CKPT_DIR / f"stable_query_vq_k{NUM_QUERY_TOKENS}_{RUN_TAG}.pt"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Model

In [ ]:
config = StableQwen3TTSConfig.from_pretrained(
    MODEL_ID,
    num_query_tokens=NUM_QUERY_TOKENS,
    anchor_num_frames=ANCHOR_FRAMES,
    sub_talker_loss_weight=SUB_TALKER_WEIGHT,
)
model = StableQwen3TTSForConditionalGeneration.from_pretrained(
    MODEL_ID,
    config=config,
    dtype=DTYPE,
    attn_implementation=ATTN_IMPLEMENTATION,
).to(DEVICE)
model.eval()

processor = Qwen3TTSProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
audio_tokenizer = processor.audio_tokenizer.to(DEVICE)

for parameter in model.parameters():
    parameter.requires_grad_(False)

init_ids = model.init_query(generator=torch.Generator(device=model.query.device).manual_seed(SEED))
model.query.requires_grad_(True)

print(f"talker hidden {config.talker_config.hidden_size}  code groups {config.talker_config.num_code_groups}")
print(f"query {tuple(model.query.shape)}  trainable {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

## Data

`build_prompt` renders one sample in one of four registers. `RAW` keeps the LibriTTS-P
persona, the other three mirror the InstructTTSEval instruction shapes. Attributes that
LibriTTS-P does not carry, notably `accent` and `emotion`, are left out rather than
invented: the judge scores accent directly and penalises any explicit mismatch, and
omission additionally teaches the partial specification that DSD and RP rely on.

In [ ]:
download_libritts_p_metadata(root="./data", annotator="df1")
dataset = LIBRITTS_P_Custom(root="./data", download=True, max_z_score=2, min_group_size=0)


def _load_audio_soundfile(self, path):
    candidates = [path, f"{path}.wav"] if not str(path).endswith(".wav") else [path]
    for candidate in candidates:
        if Path(candidate).exists():
            audio, sample_rate = soundfile.read(candidate, dtype="float32", always_2d=True)
            return torch.from_numpy(audio.T), int(sample_rate)
    raise FileNotFoundError(path)


# torchaudio.load needs torchcodec, whose ffmpeg shared objects are absent here.
dataset._load_audio = types.MethodType(_load_audio_soundfile, dataset)

waveform, rate = dataset._load_audio(dataset.data[0]["audio_path"])
print(f"dataset size {len(dataset)}  audio probe {tuple(waveform.shape)} @ {rate}")

In [ ]:
@torch.no_grad()
def encode_codec(waveform, sample_rate):
    target_rate = int(processor.feature_extractor.sampling_rate)
    audio = torch.as_tensor(np.asarray(waveform), dtype=torch.float32).squeeze()
    if int(sample_rate) != target_rate:
        audio = torch.from_numpy(librosa.resample(audio.numpy(), orig_sr=int(sample_rate), target_sr=target_rate))
    audio = audio.unsqueeze(0).to(DEVICE, dtype=audio_tokenizer.dtype)
    codes = audio_tokenizer.encode(audio, torch.ones_like(audio).bool(), return_dict=True).audio_codes[0]
    return codes.long()

In [ ]:
samples_per_frame = audio_tokenizer.get_encode_downsample_rate()
target_rate = int(processor.feature_extractor.sampling_rate)
order = list(range(len(dataset)))
random.Random(SEED).shuffle(order)

pairs, encode_started = [], time.perf_counter()

for index in order:
    if len(pairs) >= NUM_PAIRS:
        break
    sample = dataset[index]
    duration = sample["waveform"].shape[-1] / sample["sample_rate"]
    if not (MIN_CODEC_FRAMES <= duration * target_rate / samples_per_frame <= MAX_CODEC_FRAMES):
        continue
    codec = encode_codec(sample["waveform"], sample["sample_rate"])
    if not (MIN_CODEC_FRAMES <= codec.shape[0] <= MAX_CODEC_FRAMES):
        continue
    # __getitem__ drops the acoustic columns that the APS, DSD and RP renderers read.
    sample = {**dataset.data[index], **sample}
    persona, view = build_prompt(sample, epoch=0, idx=index, weights=VIEW_WEIGHTS)
    text = sample.get("normalized_text") or sample.get("original_text") or ""
    pairs.append({"persona": persona, "view": view, "codec": codec, "text": text})
    if len(pairs) % 400 == 0:
        print(f"  cached {len(pairs)}/{NUM_PAIRS}  {(time.perf_counter() - encode_started) / 60:.1f} min", flush=True)

print(f"encoding took {(time.perf_counter() - encode_started) / 60:.1f} min")

view_counts = {}
for pair in pairs:
    view_counts[pair["view"]] = view_counts.get(pair["view"], 0) + 1
print(f"cached {len(pairs)} pairs")
print("view mix:", {v.value: c for v, c in view_counts.items()})
for view in PromptView:
    example = next((p for p in pairs if p["view"] == view), None)
    if example is not None:
        print(f"\n===== {view.value} =====\n{example['persona']}")

## Teacher forcing

The prompt is the single-pass prefill: instruct, role, codec prefix, the query block, and
the anchor bos slot. The audio region is appended as summed codec embeddings, so position
`P - 1` predicts frame 0. `labels` carries the codec targets at those positions and `-100`
everywhere else.

`STREAM_TEXT_IN_TRAINING` keeps the reconstruction objective when off, which is the recipe
the earlier checkpoints were trained with. When on, the transcript streams from
`ANCHOR_FRAMES` so training matches the generation schedule exactly, at the cost of the
transcript no longer lining up with the frames it describes.

In [ ]:
STREAM_TEXT_IN_TRAINING = False


def codec_summed(codec):
    embeds = model.get_input_embeddings()(codec[:, 0])
    for group in range(config.talker_config.num_code_groups - 1):
        embeds = embeds + model.code_predictor.get_input_embeddings()[group](codec[:, group + 1])
    return embeds


def build_sequence(pair, query, tts_pad, tts_eos):
    instruct_ids = tokenizer(processor._build_instruct_text(pair["persona"]), return_tensors="pt").input_ids.to(DEVICE)
    role_ids = tokenizer(processor._build_synthesis_text(pair["text"] or "x"), return_tensors="pt").input_ids.to(DEVICE)
    codec = pair["codec"]

    prompt, _ = model.build_anchor_prompt(instruct_ids, role_ids, query=query)
    prompt_length, num_frames = prompt.shape[1], codec.shape[0]

    audio = (codec_summed(codec) + tts_pad.squeeze(0)).to(DTYPE).unsqueeze(0)
    if STREAM_TEXT_IN_TRAINING:
        stream = model.build_trailing_text(role_ids[:, 3:-5], tts_pad, tts_eos)[:, : num_frames - 1]
        if stream.shape[1] < num_frames - 1:
            stream = torch.cat([stream, tts_pad.expand(-1, num_frames - 1 - stream.shape[1], -1)], dim=1)
        audio = torch.cat([audio[:, :1], audio[:, 1:] + stream], dim=1)

    sequence = torch.cat([prompt, audio[:, :-1]], dim=1).squeeze(0)
    labels = torch.full((sequence.shape[0], codec.shape[1]), -100, dtype=torch.long, device=DEVICE)
    labels[prompt_length - 1 : prompt_length - 1 + num_frames] = codec
    return sequence, labels


def build_batch(batch, query):
    _, tts_eos, tts_pad, _ = model._talker_special_embeds(None, DTYPE)
    built = [build_sequence(pair, query, tts_pad, tts_eos) for pair in batch]
    width = max(sequence.shape[0] for sequence, _ in built)
    hidden_size = built[0][0].shape[-1]
    num_groups = built[0][1].shape[-1]

    inputs_embeds = torch.zeros(len(built), width, hidden_size, dtype=DTYPE, device=DEVICE)
    attention_mask = torch.zeros(len(built), width, dtype=torch.long, device=DEVICE)
    labels = torch.full((len(built), width, num_groups), -100, dtype=torch.long, device=DEVICE)
    # Left padding, matching the layout generation uses; get_rope_index derives positions
    # from the mask so the padded prefix carries no position.
    for index, (sequence, sequence_labels) in enumerate(built):
        length = sequence.shape[0]
        inputs_embeds[index, width - length :] = sequence
        attention_mask[index, width - length :] = 1
        labels[index, width - length :] = sequence_labels
    return inputs_embeds, attention_mask, labels


def loss_components(inputs_embeds, attention_mask, labels):
    outputs = model(
        inputs_embeds=inputs_embeds, attention_mask=attention_mask, use_cache=False, output_hidden_states=True
    )
    selected = labels[..., 0] != -100
    hidden_states = outputs.hidden_states[0][-1][selected]
    codes = labels[selected]
    ce0 = F.cross_entropy(model.codec_head(hidden_states).float(), codes[:, 0])
    sub = model.sub_talker_loss(hidden_states, codes)
    return ce0, sub, ce0 + SUB_TALKER_WEIGHT * sub

## Training

In [ ]:
optimizer = torch.optim.AdamW([model.query], lr=LEARNING_RATE)


def lr_factor(step):
    progress = min(max(step / max(1, STEPS), 0.0), 1.0)
    if SCHEDULE == "cosine":
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    if SCHEDULE == "linear":
        return 1.0 - progress
    return 1.0


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
initial_query = model.query.detach().clone()
codebook_norm = float(model.get_projected_text_vocab().norm(dim=-1).mean())


def project_to_codebook_shell():
    with torch.no_grad():
        scale = codebook_norm / model.query.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        model.query.mul_(scale.clamp_max(1.0))
sequence_lengths = [
    len(tokenizer(processor._build_instruct_text(pair["persona"])).input_ids) + 41 + int(pair["codec"].shape[0])
    for pair in pairs
]
batch_order = sorted(range(len(pairs)), key=lambda index: sequence_lengths[index])

batches, current = [], []
for index in batch_order:
    width = max([sequence_lengths[index]] + [sequence_lengths[i] for i in current])
    if current and ((len(current) + 1) * width > MAX_BATCH_TOKENS or len(current) == BATCH_SIZE):
        batches.append(current)
        current = []
    current.append(index)
if current:
    batches.append(current)
random.Random(SEED).shuffle(batches)
print(
    f"{len(batches)} batches  sizes {min(map(len, batches))}-{max(map(len, batches))}  "
    f"mean {sum(map(len, batches)) / len(batches):.1f}"
)

history, started = [], time.perf_counter()

for step in range(STEPS):
    picks = batches[step % len(batches)]
    _, query = model.quantize_query()
    ce0, sub, loss = loss_components(*build_batch([pairs[i] for i in picks], query))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    scheduler.step()
    if RUN_PROJECT:
        project_to_codebook_shell()

    history.append({"step": step, "loss": float(loss.detach()), "ce0": float(ce0.detach()), "sub": float(sub.detach())})
    if step % 25 == 0 or step == STEPS - 1:
        window = [h["loss"] for h in history[-50:]]
        drift = float((model.query.detach() - initial_query).norm(dim=-1).mean())
        with torch.no_grad():
            nearest = model.get_projected_text_vocab()[model.quantize_query()[0]]
            gap = float((model.query.detach().float() - nearest).norm(dim=-1).mean())
        elapsed = time.perf_counter() - started
        print(
            f"step {step:4d}  loss {history[-1]['loss']:7.4f}  ce0 {history[-1]['ce0']:6.4f}  "
            f"sub {history[-1]['sub']:7.4f}  avg50 {sum(window) / len(window):7.4f}  "
            f"drift {drift:6.3f}  gap {gap:6.3f}  lr {scheduler.get_last_lr()[0]:.5f}  bs {len(picks):2d}  "
            f"peak {torch.cuda.max_memory_allocated() / 2**30:.1f}G  {elapsed / 60:.1f} min"
        )

print(f"\ntrained {STEPS} steps (seed {SEED}) in {(time.perf_counter() - started) / 60:.1f} min")
print(f"first 25 avg {sum(h['loss'] for h in history[:25]) / 25:.4f}  last 25 avg {sum(h['loss'] for h in history[-25:]) / 25:.4f}")
with torch.no_grad():
    codebook = model.get_projected_text_vocab()
    nearest = codebook[model.quantize_query()[0]]
    print(
        f"query norm {float(model.query.norm(dim=-1).mean()):.3f}  "
        f"codebook norm {float(codebook.norm(dim=-1).mean()):.3f}  "
        f"gap to nearest {float((model.query.float() - nearest).norm(dim=-1).mean()):.3f}"
    )

In [ ]:
token_ids, _ = model.quantize_query()
torch.save(
    {
        "param": model.query.detach().to(torch.bfloat16).cpu(),
        "token_ids": token_ids.cpu().tolist(),
        "k": NUM_QUERY_TOKENS,
        "anchor_num_frames": ANCHOR_FRAMES,
        "qtype": "vq",
        "D": config.talker_config.hidden_size,
        "loss_history": history,
        "batch_size": BATCH_SIZE,
        "num_pairs": len(pairs),
    },
    CKPT_PATH,
)
print(f"saved {CKPT_PATH}")
print("token ids:", token_ids.cpu().tolist())

## Generation

One `generate` call produces the anchor and the utterance. `trim_anchor` drops the anchor
segment after vocoding, so the vocoder still sees the boundary in context.

In [ ]:
@torch.no_grad()
def synthesize(texts, personas, languages=None, **generation_kwargs):
    input_ids, instruct_ids = [], []
    for text, persona in zip(texts, personas):
        input_ids.append(tokenizer(processor._build_synthesis_text(text), return_tensors="pt").input_ids.to(DEVICE))
        instruct_ids.append(tokenizer(processor._build_instruct_text(persona), return_tensors="pt").input_ids.to(DEVICE))

    codes, anchor_frames = model.generate(
        input_ids=input_ids, instruct_ids=instruct_ids, languages=languages, **generation_kwargs
    )
    waveforms = [waveform.squeeze().float().cpu() for waveform in processor.batch_decode(codes)]
    upsample_rate = audio_tokenizer.get_decode_upsample_rate()
    return model.trim_anchor(waveforms, upsample_rate, anchor_frames), int(processor.feature_extractor.sampling_rate)

In [ ]:
demo_text = "The quick brown fox jumps over the lazy dog, and then it does so again."
demo_personas = [pairs[0]["persona"], pairs[1]["persona"]]
waveforms, sample_rate = synthesize([demo_text, demo_text], demo_personas)
for index, waveform in enumerate(waveforms):
    print(f"sample {index}: {waveform.shape[-1] / sample_rate:.2f}s")

## Notes

- The backbone stays frozen. Only `model.query` receives gradient, through the straight
  through estimator in `quantize_query`.
- Sampling stays at `temperature=0.9`, `top_k=50` for both the talker and the sub-talker.
  Constraining it raises the consistency metric but makes the setting a voice clone.
- `ANCHOR_FRAMES` is a fixed budget, unlike the earlier two-stage anchor which ran to its
  own codec EOS. At 12 Hz, 32 frames is about 2.7 seconds.
- Checkpoints store `token_ids`. Against a frozen backbone the parameter is recoverable
  from them through `model.init_query(token_ids)`.